# 🚀 MiniMax H3 (v3.3) 全模态参考生视频 (Ref2VA) 原生实测工作流

> **实验目的**：专注于**多模态解耦提取、运镜轨迹迁移与音画双轨分流**。  
> **核心技术原理**：
> 1. **全模态硬性配额门禁**：混合输入总文件 $\le 12$（图片 $\le 9$、视频 $\le 3$、音频 $\le 3$；单段 $2\sim 15\text{s}$，累计时长 $\le 15.0\text{s}$）。**音频红线**：严禁提交纯音频，必须搭配图片或视频！
> 2. **上游官方拓扑规范**：`LoadVideo` (输入 `file`) 输出单个 `VIDEO`，通过 `GetVideoComponents` 解包为 `images` (slot 0) 与 `audio` (slot 1)。
> 3. **Autogrow 端口规范**：`MiniMaxH3ReferenceToVideo` 使用 `ref_image_size: "match"`, `ref_images.ref_image_0`, `ref_videos.ref_video_0`, `ref_video_audios.ref_video_audio_0`, `ref_audios.ref_audio_0`。
> 4. **提示词双向兼容**：支持 `@图片1` / `@视频1` / `@音频1` 与 `<Picture 1>` / `<Video 1>` / `<Audio 1>`。视频自带音轨优先占位编号 1。
> 5. **L4 黄金生产规格**：`960x544`、`124` 帧、Turbo 4-Step，BasicGuider (CFG=1.0) 单路引导。

---
### 🔄 即插即用（Swap & Run）快速指引：
1. **开箱即用**：直接点击「全部运行」，脚本内置橙色小车图、24fps 横向行驶参考视频与 32kHz 电机声，**零上传即可一键跑通**！
2. **替换自己的素材**：在【步骤 4】中填入 `CUSTOM_REF_IMAGE`、`CUSTOM_REF_VIDEO`、`CUSTOM_REF_AUDIO` 即可无缝切换！

### 步骤 1：运行配置与算力看门狗守护

In [ ]:
# ==================== 步骤 1：运行配置与 2.0 小时算力看门狗守护 ====================
MODE = "ref2va"                  # 当前实验控制流模式: "ref2va"
TURBO_STEPS = 4                   # 采样步数: 8 (I2V/FLF2V 推荐 8 步高保真; Ref2VA 推荐 4 步)
AUTO_SHUTDOWN = True             # 全部完成后自动执行 runtime.unassign() 释放实例停止扣费
WATCHDOG_HOURS = 2.0             # 算力看门狗硬熔断上限（小时），防止异常卡死消耗算力

BACKUP_WEIGHTS_TO_DRIVE = False  # 默认直拉 NVMe；需要持久化备份时开启
ALLOW_DRIVE_FALLBACK = True       # HF 下载失败时尝试完整 Drive 缓存

DIFFUSION_VARIANT = "auto"       # 权重选型: "auto" (L4 下自动匹配 fp8_scaled 消除慢回退)
SAMPLER_NAME = "res_multistep"   # 官方推荐采样器: "res_multistep" (或 "euler")

import os, sys, time, json, shutil, threading, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_MODELS = DRIVE_ROOT / "MiniMax_Models"
DRIVE_OUTPUTS = DRIVE_ROOT / "MiniMax_Outputs"

COMFY_ROOT = Path("/content/ComfyUI")
LOCAL_MODELS = COMFY_ROOT / "models"
COMFY_MODELS = LOCAL_MODELS
LOCAL_OUT = COMFY_ROOT / "output"
COMFY_LOG = Path("/content/comfyui.log")
SERVER_URL = "http://127.0.0.1:8188"

def log(msg):
    print(time.strftime("[%H:%M:%S] ") + str(msg), flush=True)

def safe_shutdown(reason="正常完成"):
    log(f"🛑 触发 Colab 运行时安全释放: {reason}")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        log("💾 Drive 缓存已刷盘")
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        log(f"⚠️ runtime.unassign() 异常: {e}，执行容器级退出")
        os.system("sync")
        os.kill(1, 15)

if not globals().get("_WATCHDOG_RUNNING"):
    def _watchdog_timer():
        time.sleep(int(WATCHDOG_HOURS * 3600))
        log(f"🚨 [WATCHDOG 熔断] 运行时已超 {WATCHDOG_HOURS} 小时，自动断开保护算力！")
        safe_shutdown("看门狗超时熔断")
    threading.Thread(target=_watchdog_timer, daemon=True).start()
    _WATCHDOG_RUNNING = True
    log(f"🛡️ 算力看门狗守门员已就绪：{WATCHDOG_HOURS} 小时超时自动断开")

### 步骤 2：挂载 Google Drive (官方原生接口)

In [ ]:
# ==================== 步骤 2：挂载 Google Drive (官方原生合规接口) ====================
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    log("⏳ 正在通过官方接口挂载 Google Drive（请在弹出框中点击授权）...")
    drive.mount("/content/drive")
else:
    log("✅ Google Drive 处于已挂载就绪状态")

DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
log(f"📁 云端模型持久化缓存: {DRIVE_MODELS}")
log(f"🎬 成片与日志最终落盘: {DRIVE_OUTPUTS}")

### 步骤 3：硬件门禁检测 (验证 L4 GPU)

In [ ]:
# ==================== 步骤 3：硬件门禁检测 (锁定 L4 GPU，零内核显存占用) ====================
def check_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
            text=True
        ).strip().splitlines()[0]
        name, cc_str, total_mb = [x.strip() for x in out.split(",")]
        return name, float(cc_str), float(total_mb) / 1024
    except Exception as e:
        raise RuntimeError(f"未检测到可用 GPU，请在「更改运行时类型」中选择 L4 GPU: {e}")

gpu_name, cc, total_gb = check_gpu()
disk_free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
log(f"🖥️ 当前显卡: {gpu_name} (算力 {cc:.1f}) | 显存: {total_gb:.2f} GB | 本地磁盘剩余: {disk_free_gb:.1f} GB")

if cc < 8.0:
    log(f"🚨 致命拦截：{gpu_name} 算力 {cc:.1f} < 8.0，不支持量化算子加速！")
    safe_shutdown("硬件不达标 (非 L4)")
    raise RuntimeError(f"硬件不达标 ({gpu_name})，已自动断开。请重新分配 L4 实例。")

log("🎉 硬件门禁验证通过，允许运行！")

### 步骤 4：🔄 替换素材与提示词 (Swap & Run)

In [ ]:
# ==================== 步骤 4：🔄 替换素材与提示词 (Swap & Run 核心单元格) ====================
CUSTOM_REF_IMAGE = ""  # 留空生成橙色小车参考图
CUSTOM_REF_VIDEO = ""  # 留空生成小车从左向右行驶的 24fps 参考视频
CUSTOM_REF_AUDIO = ""  # 留空生成低频电机声
CUSTOM_PROMPT = ""

DEFAULT_PROMPT = (
    "A small orange toy delivery rover with two black wheels and a cyan windshield, "
    "matching <Picture 1>, drives steadily from the left side to the right side of the frame "
    "following the clear horizontal movement in <Video 1>. "
    "Its wheels rotate as it travels across the floor. Fixed wide camera, full vehicle visible, "
    "continuous visible movement throughout the shot. "
    "Soft motor hum matching <Audio 2>, with the quiet background ambience of <Audio 1>."
)

from PIL import Image, ImageDraw
import subprocess, shutil, math

LOCAL_ASSETS_DIR = Path("/content/assets")
LOCAL_ASSETS_DIR.mkdir(parents=True, exist_ok=True)
REF_IMAGE_PATH = LOCAL_ASSETS_DIR / "ref2va_rover_anchor.png"
REF_VIDEO_PATH = LOCAL_ASSETS_DIR / "ref2va_rover_motion_24fps.mp4"
REF_AUDIO_PATH = LOCAL_ASSETS_DIR / "ref2va_motor_audio.wav"

def generate_default_ref2va_suite():
    w, h, fps, frames = 960, 544, 24, 124
    duration = frames / fps
    def draw_frame(index):
        img = Image.new("RGB", (w, h), (205, 220, 230))
        draw = ImageDraw.Draw(img)
        draw.rectangle((0, 380, w, h), fill=(150, 160, 170))
        for x in range(0, w, 120):
            draw.line((x, 380, x, h), fill=(120, 130, 140), width=2)
        cx = 170 + round(620 * index / (frames - 1))
        draw.ellipse((cx-105, 398, cx+105, 426), fill=(110, 120, 130))
        draw.rounded_rectangle((cx-100, 285, cx+100, 385), radius=24, fill=(245, 125, 30), outline=(120, 65, 20), width=4)
        draw.rounded_rectangle((cx-60, 230, cx+55, 310), radius=16, fill=(245, 125, 30), outline=(120, 65, 20), width=4)
        draw.rectangle((cx-40, 248, cx+35, 280), fill=(30, 170, 210))
        draw.rectangle((cx+80, 320, cx+99, 340), fill=(255, 245, 180))
        for wheel_x in (cx-62, cx+62):
            draw.ellipse((wheel_x-30, 355, wheel_x+30, 415), fill=(25, 30, 35))
            angle = index * 0.23
            dx, dy = 22*math.cos(angle), 22*math.sin(angle)
            draw.line((wheel_x-dx, 385-dy, wheel_x+dx, 385+dy), fill=(210, 215, 220), width=5)
        return img

    draw_frame(0).save(REF_IMAGE_PATH)
    ffmpeg_bin = shutil.which("ffmpeg") or "ffmpeg"
    subprocess.run([
        ffmpeg_bin, "-y", "-v", "error", "-f", "lavfi", "-i",
        f"sine=frequency=90:duration={duration}", "-ar", "32000", "-ac", "2",
        "-c:a", "pcm_s16le", str(REF_AUDIO_PATH)
    ], check=True)
    # 逐帧绘制真实位移，不依赖 drawbox 对时间表达式的支持。
    video_frames = LOCAL_ASSETS_DIR / "rover_frames"
    video_frames.mkdir(exist_ok=True)
    for index in range(frames):
        draw_frame(index).save(video_frames / f"{index:04d}.png")
    subprocess.run([
        ffmpeg_bin, "-y", "-v", "error", "-framerate", str(fps),
        "-i", str(video_frames / "%04d.png"), "-f", "lavfi", "-i",
        f"sine=frequency=180:duration={duration}", "-frames:v", str(frames),
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
        "-ar", "32000", "-ac", "2", "-movflags", "+faststart", str(REF_VIDEO_PATH)
    ], check=True)
    log("默认素材已生成：橙色小车由左向右移动 620 像素，124 帧，附带音轨")

custom_paths = [CUSTOM_REF_IMAGE, CUSTOM_REF_VIDEO, CUSTOM_REF_AUDIO]
if any(custom_paths):
    if not all(custom_paths):
        raise ValueError("自定义模式请同时填写图片、视频和音频路径；默认模式请全部留空")
    for value in custom_paths:
        if not Path(value).is_file():
            raise FileNotFoundError(f"参考素材不存在: {value}")
    ACTIVE_IMG, ACTIVE_VID, ACTIVE_AUD = map(Path, custom_paths)
else:
    generate_default_ref2va_suite()
    ACTIVE_IMG, ACTIVE_VID, ACTIVE_AUD = REF_IMAGE_PATH, REF_VIDEO_PATH, REF_AUDIO_PATH

# 推理前先确认输入参考视频确实有可见运动。
from IPython.display import Video, display
log("输入参考预览：应看到橙色小车从左向右行驶；生成结果需另行比较")
display(Video(str(ACTIVE_VID), embed=True))

import re
def normalize_h3_tags(prompt_str):
    p = re.sub(r'[@<]图片(\d+)[>]?', r'<Picture \1>', prompt_str)
    p = re.sub(r'[@<]视频(\d+)[>]?', r'<Video \1>', p)
    p = re.sub(r'[@<]音频(\d+)[>]?', r'<Audio \1>', p)
    return p

ACTIVE_PROMPT = normalize_h3_tags(CUSTOM_PROMPT if CUSTOM_PROMPT.strip() else DEFAULT_PROMPT)
log(f"📝 规范化提示词: {ACTIVE_PROMPT[:90]}...")

### 步骤 5：输入配额门禁校验与 24fps 恒定帧率转码

In [ ]:
# ==================== 步骤 5：输入配额门禁校验与 24fps 恒定帧率转码 ====================
def validate_ref2va_quotas(images, videos, audios):
    if len(audios) > 0 and len(images) + len(videos) == 0:
        raise ValueError("Ref2VA 硬性红线拦截：音频绝不可作为唯一输入独立提交！")
    if len(images) > 9:
        raise ValueError(f"图片超标: {len(images)} > 9")
    if len(videos) > 3:
        raise ValueError(f"视频超标: {len(videos)} > 3")
    if len(audios) > 3:
        raise ValueError(f"音频超标: {len(audios)} > 3")
    if len(images) + len(videos) + len(audios) > 12:
        raise ValueError("总文件超标: > 12")
    log(f"🛡️ 配额门禁验证通过: 图片 {len(images)}/9 | 视频 {len(videos)}/3 | 音频 {len(audios)}/3 | 合计 {len(images)+len(videos)+len(audios)}/12")

validate_ref2va_quotas([ACTIVE_IMG], [ACTIVE_VID], [ACTIVE_AUD])

COMFY_INPUT = COMFY_ROOT / "input"
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
STAGED_IMG = COMFY_INPUT / "ref2va_img_staged.png"
STAGED_VID = COMFY_INPUT / "ref2va_vid_24fps_staged.mp4"
STAGED_AUD = COMFY_INPUT / "ref2va_aud_staged.wav"

shutil.copy2(ACTIVE_IMG, STAGED_IMG)
shutil.copy2(ACTIVE_AUD, STAGED_AUD)

subprocess.run([
    "ffmpeg", "-y", "-i", str(ACTIVE_VID),
    "-vf", "scale=960:544:flags=lanczos,fps=fps=24:round=near,format=yuv420p",
    "-c:v", "libx264", "-r", "24", "-colorspace", "bt709",
    "-c:a", "aac", "-ar", "32000", "-ac", "2",
    str(STAGED_VID)
], capture_output=True, check=True)

log(f"✅ 24fps CFR 视频转码完成并就绪: {STAGED_VID.name}")

### 步骤 6：安装 ComfyUI 核心环境与 PyAV

In [ ]:
# ==================== 安装 ComfyUI 核心依赖与 PyAV ====================
# 素材准备已创建 COMFY_ROOT/input，不能直接 git clone 到非空目录。
# 先克隆至临时目录，成功后合并代码，保留已有 input/models/output。
import tempfile
if not (COMFY_ROOT / "main.py").exists():
    log("📦 正在克隆官方 ComfyUI 仓库...")
    with tempfile.TemporaryDirectory(prefix="comfy-install-", dir=str(COMFY_ROOT.parent)) as tmp:
        checkout = Path(tmp) / "ComfyUI"
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(checkout)],
            capture_output=True, text=True,
        )
        if result.returncode:
            raise RuntimeError(f"ComfyUI 克隆失败，Git 原始错误：\n{result.stderr}")
        COMFY_ROOT.mkdir(parents=True, exist_ok=True)
        for entry in checkout.iterdir():
            target = COMFY_ROOT / entry.name
            if entry.is_dir():
                shutil.copytree(entry, target, dirs_exist_ok=True)
            else:
                shutil.copy2(entry, target)
else:
    log("✅ 检测到 ComfyUI 代码已就绪")

# 每次确保依赖安装完成，支持上次安装中断后重跑。
log("📦 正在检查安装 ComfyUI 基础依赖...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_ROOT / "requirements.txt")], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "huggingface_hub", "safetensors", "accelerate", "Pillow", "tqdm"], check=True)
log("✅ ComfyUI 核心与 PyAV 依赖就绪！")

### 步骤 7：模型安全挂载与直连校验 (Ref2VA 权重)

In [ ]:
# ==================== 模型 NVMe 优先加载与完整性校验 (精确字节 + 头部双校验) ====================
import struct
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import enable_progress_bars
from tqdm.auto import tqdm
enable_progress_bars()

EXACT_SIZES = {
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20970379616,
    "diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20958205608,
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20970379616,
    "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20958205608,
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15687142551,
    "vae/minimax_h3_video_vae_int8_convrot.safetensors": 2811065184,
    "vae/minimax_h3_video_vae_fp16.safetensors": 5207808496,
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605254808,
    "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors": 1956193000,
    "loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors": 1956192992,
    "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors": 1956193000,
}

def resolve_variant_and_files():
    cuda_ver = None
    try:
        import torch
        if torch.cuda.is_available() and hasattr(torch.version, "cuda") and torch.version.cuda:
            cuda_ver = torch.version.cuda
    except Exception:
        pass

    if DIFFUSION_VARIANT == "auto":
        if cuda_ver and cuda_ver.startswith("13."):
            actual_variant = "int8_convrot"
        else:
            actual_variant = "fp8_scaled"
    else:
        actual_variant = DIFFUSION_VARIANT

    mode = "ref2va"
    if mode == "ref2va":
        diff_file = f"minimax_h3_ref2va_pruned_{actual_variant}.safetensors"
        lora_file = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
    else:
        diff_file = f"minimax_h3_fl2va_pruned_{actual_variant}.safetensors"
        lora_file = "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors" if 4 == 8 else "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"

    vae_file = "minimax_h3_video_vae_int8_convrot.safetensors" if actual_variant == "int8_convrot" else "minimax_h3_video_vae_fp16.safetensors"
    return actual_variant, diff_file, vae_file, lora_file

ACTUAL_VARIANT, DIFFUSION_FILE, VIDEO_VAE_FILE, LORA_FILE = resolve_variant_and_files()

MODELS_SPEC = [
    (f"diffusion_models/{DIFFUSION_FILE}", "diffusion_models", EXACT_SIZES[f"diffusion_models/{DIFFUSION_FILE}"]),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders", EXACT_SIZES["text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"]),
    (f"vae/{VIDEO_VAE_FILE}", "vae", EXACT_SIZES[f"vae/{VIDEO_VAE_FILE}"]),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae", EXACT_SIZES["vae/minimax_h3_audio_vae_fp32.safetensors"]),
    (f"loras/{LORA_FILE}", "loras", EXACT_SIZES[f"loras/{LORA_FILE}"]),
]

def check_safetensors_file(file_path: Path, expected_size: int) -> bool:
    if not file_path.is_file():
        return False
    try:
        if file_path.stat().st_size != expected_size:
            return False
        with open(file_path, "rb") as f:
            header_bytes = f.read(8)
            if len(header_bytes) < 8:
                return False
            hlen = struct.unpack("<Q", header_bytes)[0]
            if not 0 < hlen <= min(expected_size - 8, 100 * 1024 * 1024):
                return False
            header = json.loads(f.read(hlen).decode("utf-8"))
        max_end = max(v["data_offsets"][1] for k, v in header.items() if k != "__metadata__")
        return (8 + hlen + max_end) == expected_size
    except Exception:
        return False

log("=" * 68)
log(f"📦 MiniMax H3 权重挂载检测 (模式: REF2VA) | 选型: {ACTUAL_VARIANT}")
log("=" * 68)

def copy_with_progress(src, dst, desc):
    dst.parent.mkdir(parents=True, exist_ok=True)
    partial = dst.with_name(dst.name + ".partial")
    try:
        with src.open("rb") as reader, partial.open("wb") as writer, tqdm(
            total=src.stat().st_size, unit="B", unit_scale=True,
            unit_divisor=1024, desc=desc
        ) as progress:
            while True:
                chunk = reader.read(32 * 1024 * 1024)
                if not chunk:
                    break
                writer.write(chunk)
                progress.update(len(chunk))
        partial.replace(dst)
    finally:
        partial.unlink(missing_ok=True)


def ensure_weight_fast(rel_path, subfolder, exact_bytes):
    fname = Path(rel_path).name
    local_file = LOCAL_MODELS / subfolder / fname
    drive_file = DRIVE_MODELS / subfolder / fname
    local_file.parent.mkdir(parents=True, exist_ok=True)
    if not local_file.is_symlink() and check_safetensors_file(local_file, exact_bytes):
        log(f"✅ 本地 NVMe 已就绪: {fname}")
        return local_file

    # 避免旧软链接把下载或拷贝重定向到 Drive。
    if local_file.is_symlink() or local_file.exists():
        local_file.unlink()
    free_bytes = shutil.disk_usage(LOCAL_MODELS).free
    if free_bytes < exact_bytes:
        raise RuntimeError(f"NVMe 空间不足: {fname} 需要 {exact_bytes} 字节，可用 {free_bytes}")
    try:
        log(f"⬇️ HF 直拉到 NVMe: {rel_path} ({exact_bytes / 1024**3:.2f} GiB)")
        downloaded = Path(hf_hub_download(
            repo_id="Comfy-Org/MiniMax-H3", filename=rel_path,
            local_dir=str(LOCAL_MODELS / "_hf_cache"),
        ))
        if not check_safetensors_file(downloaded, exact_bytes):
            downloaded.unlink(missing_ok=True)
            raise RuntimeError(f"HF 权重完整性校验失败: {fname}")
        shutil.move(str(downloaded), str(local_file))
    except Exception as download_error:
        if not ALLOW_DRIVE_FALLBACK or not check_safetensors_file(drive_file, exact_bytes):
            raise RuntimeError(f"无法加载权重 {fname}: HF 下载失败且无可用 Drive 回退") from download_error
        log(f"⚠️ HF 下载失败: {download_error}; 从 Drive 恢复 {fname}")
        copy_with_progress(drive_file, local_file, "Drive → NVMe")
        if not check_safetensors_file(local_file, exact_bytes):
            local_file.unlink(missing_ok=True)
            raise RuntimeError(f"Drive 恢复校验失败: {fname}")

    if BACKUP_WEIGHTS_TO_DRIVE and not check_safetensors_file(drive_file, exact_bytes):
        try:
            copy_with_progress(local_file, drive_file, "NVMe → Drive 可选备份")
        except Exception as backup_error:
            log(f"⚠️ Drive 备份失败，本地权重仍可使用: {backup_error}")
    log(f"✅ 权重通过精确字节与 safetensors 头部校验: {fname}")
    return local_file


for rel_path, subfolder, exact_bytes in MODELS_SPEC:
    ensure_weight_fast(rel_path, subfolder, exact_bytes)
log("🎉 全部 5 个模型权重已就绪于本地 NVMe")

### 步骤 8：启动 ComfyUI 守护服务与前置预检

In [ ]:
# ==================== 启动 ComfyUI 守护服务与前置节点预检 ====================
import urllib.request, urllib.error

def post_json(path, payload=None, timeout=10):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(f"{SERVER_URL}{path}", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        content = resp.read().decode("utf-8")
        return json.loads(content) if content.strip() else {}

def is_comfy_ready():
    try:
        res = post_json("/system_stats", timeout=3)
        return "devices" in res
    except Exception:
        return False

subprocess.run(["pkill", "-9", "-f", "main.py.*8188"], check=False)
time.sleep(2)

log("🚀 正在启动 ComfyUI 后台守护服务 (--lowvram)...")
log_fp = open(COMFY_LOG, "w", encoding="utf-8")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
    cwd=str(COMFY_ROOT), stdout=log_fp, stderr=subprocess.STDOUT, start_new_session=True
)

for i in range(60):
    if comfy_proc.poll() is not None:
        tail = COMFY_LOG.read_text(errors="replace")[-2000:] if COMFY_LOG.exists() else ""
        raise RuntimeError(f"ComfyUI 提前异常退出，错误日志:\n{tail}")
    if is_comfy_ready():
        log(f"✅ ComfyUI 守护服务就绪 (PID: {comfy_proc.pid})")
        break
    time.sleep(2)
else:
    raise RuntimeError("ComfyUI 在 120 秒内未能就绪，请检查 /content/comfyui.log")

log("🔍 正在预检 ComfyUI /object_info 契约...")
OBJ_INFO = post_json("/object_info", timeout=60)
log("✅ ComfyUI 核心环境与契约预检完成！")

### 步骤 9：执行 Ref2VA 单目的生产渲染

In [ ]:
# ==================== 步骤 8：执行 Ref2VA 单目的生产渲染 (GetVideoComponents + Autogrow) ====================
import copy

BASE_WORKFLOW = json.loads(r"""{
  "1": {
    "inputs": {
      "unet_name": "minimax_h3_ref2va_pruned_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {
      "title": "MiniMax H3 Ref2VA Diffusion Model (Pruned Int8)"
    }
  },
  "2": {
    "inputs": {
      "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
      "type": "minimax",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {
      "title": "MiniMax H3 Text Encoder (NVFP4)"
    }
  },
  "4": {
    "inputs": {
      "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {
      "title": "MiniMax H3 Video VAE"
    }
  },
  "11": {
    "inputs": {
      "vae_name": "minimax_h3_audio_vae_fp32.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {
      "title": "MiniMax H3 Audio VAE (32kHz Stereo)"
    }
  },
  "10": {
    "inputs": {
      "lora_name": "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
      "strength_model": 1.0,
      "model": [
        "1",
        0
      ]
    },
    "class_type": "LoraLoaderModelOnly",
    "_meta": {
      "title": "Turbo Ref2V 4-Step LoRA"
    }
  },
  "20": {
    "inputs": {
      "image": "ref2va_subject_anchor.png",
      "upload": "image"
    },
    "class_type": "LoadImage",
    "_meta": {
      "title": "Load Reference Image 0 (Subject Anchor)"
    }
  },
  "21": {
    "inputs": {
      "file": "ref2va_motion_ref_24fps.mp4"
    },
    "class_type": "LoadVideo",
    "_meta": {
      "title": "Load Reference Video 0"
    }
  },
  "23": {
    "inputs": {
      "video": [
        "21",
        0
      ]
    },
    "class_type": "GetVideoComponents",
    "_meta": {
      "title": "Get Video Components (Frames, Audio, FPS)"
    }
  },
  "22": {
    "inputs": {
      "audio": "ref2va_pulse_audio.wav",
      "upload": "audio"
    },
    "class_type": "LoadAudio",
    "_meta": {
      "title": "Load Reference Audio 0 (Independent Pulse Audio)"
    }
  },
  "6": {
    "inputs": {
      "prompt": "The camera replicates the precise 24fps orbiting push tracking of <Video 1>. The subject preserves the character identity, hair, and costume styling of <Picture 1>. Environmental soundscape syncs with the room ambience of <Audio 1>, layered with the deep sub-bass rhythmic pulse from <Audio 2>.",
      "width": 960,
      "height": 544,
      "length": 124,
      "clip": [
        "2",
        0
      ],
      "vae": [
        "4",
        0
      ],
      "audio_vae": [
        "11",
        0
      ],
      "ref_image_size": "match",
      "ref_images.ref_image_0": [
        "20",
        0
      ],
      "ref_videos.ref_video_0": [
        "23",
        0
      ],
      "ref_video_audios.ref_video_audio_0": [
        "23",
        1
      ],
      "ref_audios.ref_audio_0": [
        "22",
        0
      ]
    },
    "class_type": "MiniMaxH3ReferenceToVideo",
    "_meta": {
      "title": "MiniMax H3 ReferenceToVideo (Multi-Ref Conditioning)"
    }
  },
  "7": {
    "inputs": {
      "model": [
        "10",
        0
      ],
      "conditioning": [
        "6",
        0
      ]
    },
    "class_type": "BasicGuider",
    "_meta": {
      "title": "BasicGuider (No CFG, CFG=1.0)"
    }
  },
  "14": {
    "inputs": {
      "noise_seed": 303
    },
    "class_type": "RandomNoise",
    "_meta": {
      "title": "Random Noise"
    }
  },
  "15": {
    "inputs": {
      "sampler_name": "res_multistep"
    },
    "class_type": "KSamplerSelect",
    "_meta": {
      "title": "KSamplerSelect (res_multistep)"
    }
  },
  "16": {
    "inputs": {
      "steps": 4,
      "denoise": 1.0,
      "scheduler": "simple",
      "model": [
        "10",
        0
      ]
    },
    "class_type": "BasicScheduler",
    "_meta": {
      "title": "BasicScheduler (simple)"
    }
  },
  "3": {
    "inputs": {
      "noise": [
        "14",
        0
      ],
      "guider": [
        "7",
        0
      ],
      "sampler": [
        "15",
        0
      ],
      "sigmas": [
        "16",
        0
      ],
      "latent_image": [
        "6",
        1
      ]
    },
    "class_type": "SamplerCustomAdvanced",
    "_meta": {
      "title": "SamplerCustomAdvanced"
    }
  },
  "8": {
    "inputs": {
      "samples": [
        "3",
        0
      ],
      "vae": [
        "4",
        0
      ]
    },
    "class_type": "VAEDecode",
    "_meta": {
      "title": "VAE Decode Video"
    }
  },
  "12": {
    "inputs": {
      "samples": [
        "3",
        0
      ],
      "vae": [
        "11",
        0
      ]
    },
    "class_type": "VAEDecodeAudio",
    "_meta": {
      "title": "VAE Decode Audio (Native 32kHz Stereo)"
    }
  },
  "13": {
    "inputs": {
      "images": [
        "8",
        0
      ],
      "audio": [
        "12",
        0
      ],
      "fps": 24.0
    },
    "class_type": "CreateVideo",
    "_meta": {
      "title": "Create Video from Frames & Audio"
    }
  },
  "9": {
    "inputs": {
      "video": [
        "13",
        0
      ],
      "filename_prefix": "H3_V3_3_Ref2VA",
      "format": "mp4",
      "codec": "auto"
    },
    "class_type": "SaveVideo",
    "_meta": {
      "title": "Save Production MP4 Video"
    }
  }
}
""")

wf = copy.deepcopy(BASE_WORKFLOW)
wf["1"]["inputs"]["unet_name"] = DIFFUSION_FILE
wf["4"]["inputs"]["vae_name"] = VIDEO_VAE_FILE
wf["10"]["inputs"]["lora_name"] = LORA_FILE

# 注入解包拓扑与 Autogrow 端口
wf["20"]["inputs"]["image"] = STAGED_IMG.name
wf["21"]["inputs"]["file"] = STAGED_VID.name
if "23" in wf:
    wf["23"]["inputs"]["video"] = ["21", 0]
wf["22"]["inputs"]["audio"] = STAGED_AUD.name

wf["6"]["inputs"]["prompt"] = ACTIVE_PROMPT
wf["6"]["inputs"]["width"] = 960
wf["6"]["inputs"]["height"] = 544
wf["6"]["inputs"]["length"] = 124
wf["6"]["inputs"]["ref_image_size"] = "match"
wf["6"]["inputs"]["ref_images.ref_image_0"] = ["20", 0]
wf["6"]["inputs"]["ref_videos.ref_video_0"] = ["23", 0]
wf["6"]["inputs"]["ref_video_audios.ref_video_audio_0"] = ["23", 1]
wf["6"]["inputs"]["ref_audios.ref_audio_0"] = ["22", 0]

wf["16"]["inputs"]["steps"] = TURBO_STEPS
wf["9"]["inputs"]["filename_prefix"] = "H3_V3_3_Ref2VA_Showcase"

log("🚀 正在提交 Ref2VA 全模态工作流至 ComfyUI 队列...")
submit = post_json("/prompt", {"prompt": wf, "client_id": "ref2va_worker"}, timeout=15)
if "error" in submit or submit.get("node_errors"):
    raise RuntimeError(f"ComfyUI 拒绝执行: {submit}")

pid = submit["prompt_id"]
log(f"   已推入队列 (Prompt ID: {pid})，正在去噪渲染...")

t0 = time.time()
while time.time() - t0 < 900:
    hist = post_json(f"/history/{pid}", timeout=5)
    if pid in hist:
        break
    time.sleep(3)
else:
    post_json("/interrupt", {})
    raise TimeoutError("渲染超时！")

cost_s = time.time() - t0
log(f"✅ Ref2VA 渲染成功！耗时: {cost_s:.1f} 秒 | 折合算力消耗: ~{1.94 * (cost_s / 3600):.4f} CU")

out_files = list(LOCAL_OUT.glob("H3_V3_3_Ref2VA_Showcase*.mp4"))
if not out_files:
    raise RuntimeError("未在输出目录找到生成的 MP4 文件！")

FINAL_MP4 = out_files[-1]
DRIVE_MP4 = DRIVE_OUTPUTS / FINAL_MP4.name
shutil.copy2(FINAL_MP4, DRIVE_MP4)
log(f"💾 成片已安全同步至 Google Drive: {DRIVE_MP4.name}")

### 步骤 10：HTML5 播放器预览与音画双轨分流检视

In [ ]:
# ==================== 步骤 9：HTML5 播放器预览与音画双轨分流检视 ====================
import base64
from IPython.display import HTML, display

with open(FINAL_MP4, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")

with open(STAGED_IMG, "rb") as f:
    b64_img = base64.b64encode(f.read()).decode("utf-8")

html = f"""
<div style="background: #111827; padding: 20px; border-radius: 12px; border: 1px solid #06b6d4; max-width: 860px; font-family: sans-serif; color: #e5e7eb;">
    <h3 style="color: #22d3ee; margin-top: 0;">🎬 MiniMax H3 V3.3 Ref2VA 全模态成片就绪</h3>
    <div style="display: flex; gap: 14px; margin-bottom: 12px;">
        <div style="flex: 1;">
            <p style="margin: 0 0 6px 0; font-size: 12px; color: #9ca3af;"><b>外观参考 (<Picture 1>):</b></p>
            <img src="data:image/png;base64,{b64_img}" style="width: 100%; border-radius: 6px; border: 1px solid #374151;" />
        </div>
        <div style="flex: 2;">
            <p style="margin: 0 0 6px 0; font-size: 12px; color: #22d3ee;"><b>运镜复刻与双音轨合成 (960x544 @ 24fps):</b></p>
            <video width="100%" controls autoplay loop style="border-radius: 6px; border: 2px solid #06b6d4;">
                <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
            </video>
        </div>
    </div>
    <p style="font-size: 12px; color: #9ca3af; margin: 0;">
        🔊 <b>分轨说明</b>：视频自带环境声景优先占位 `<Audio 1>`，独立低频脉冲占位 `<Audio 2>`，双轨经 VAE 解码自动合流成 32kHz 原生立体声。
    </p>
</div>
"""
display(HTML(html))

### 步骤 11：安全收尾与自动销毁实例 (防扣费)

In [ ]:
# ==================== 步骤 10：安全收尾与自动销毁实例 (防扣费) ====================
if AUTO_SHUTDOWN:
    log("🛑 批处理任务全部完成，正在自动销毁实例以终止算力扣费...")
    safe_shutdown("生产出片完毕，正常释放")
else:
    log("💡 提示：AUTO_SHUTDOWN 设为 False，请在右上角手动断开并释放运行时。")